# T1 — 정형 신경망 (임베딩-MLP) · 버그 회피 설계

> **질문:** GBDT 계열은 뭘 바꿔도 0.7395/ρ0.99로 포화(3모델·DART·랭킹 전부 공정 네거티브). **함수 클래스가 다른 NN은 (a) 강하면서 decorrelated해 블렌드를 미는가, (b) 같은 0.74로 천장을 확정하는가?** 둘 다 가치.
>
> **버그 회피:** 실험 1회 4시간+ 경험 → **Phase 0 프로토타입(서브샘플·소수 epoch)으로 풀런 전에 리스크를 깐다.**
>
> - **Phase 0** (~30분): 서브샘플 ~40k·단일 holdout·소수 epoch. **게이트 P: val AUC ≥ 0.72 면 풀런, < 0.70/정체면 보류.**
> - **Phase 1**: Phase 0 통과 시에만 풀 5-fold. fold-내부 전처리(대치·스케일·임베딩 train만 fit), keep(33) 입력, 동일 fold(seed42).
> - **Phase 2 게이트**: 단일 OOF≥0.735 **AND** 트리와 ρ≤0.93 **AND** (blend+NN−blend3) 증분 CI>0. NN은 시드분산 크니 채택 직전 멀티시드.
>
> **★ GPU 세션 필수.** (25만 행 NN은 CPU로 시간 내 불가.)

## 1. 셋업 + GPU 확인

In [1]:
import re, os, glob, json, warnings, time
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE=="cpu":
    print("⚠️ GPU 미감지 — 케글 Settings > Accelerator > GPU 켜고 재실행 권장(CPU는 매우 느림).")
def kfont():
    try:
        import koreanize_matplotlib; return 1
    except Exception:
        matplotlib.rcParams["axes.unicode_minus"]=False; return 0
kfont()

torch 2.10.0+cu128 | device: cuda


0

## 2. 데이터 + 설정

In [2]:
csvs=sorted(glob.glob("/kaggle/input/**/*.csv",recursive=True)) or sorted(glob.glob("**/*.csv",recursive=True))
def pick(*ks):
    for p in csvs:
        if all(k in os.path.basename(p).lower() for k in ks): return p
train=pd.read_csv(pick("train") or csvs[0]); test=pd.read_csv(pick("test"))
sp=pick("sample") or pick("submission"); sample_sub=pd.read_csv(sp) if sp and "test" not in os.path.basename(sp).lower() else None
print("train",train.shape,"| test",test.shape)

SEED=42; N_SPLITS=5
# Phase 0 프로토타입
PROTO_N=40000; PROTO_EPOCHS=8; PROTO_GATE=0.72
# Phase 1 풀런
EPOCHS=40; PATIENCE=6; BATCH=2048; LR=1e-3; HIDDEN=(256,128); DROPOUT=0.2
# Phase 2 게이트
GATE_RHO=0.93; GATE_AUC=0.735; BOOT_N=1000
RUN_MULTISEED=True   # 채택 직전에만 True (NN 시드분산 확인)

# keep 셋 (diagnostics 산출). 없으면 전체 피처.
kp=glob.glob("/kaggle/input/**/prune_list.json",recursive=True)+glob.glob("prune_list.json")+glob.glob("/kaggle/working/prune_list.json")
KEEP=json.load(open(kp[0]))["keep"] if kp else None
print("keep 로드:", (str(len(KEEP))+"개" ) if KEEP else "없음 → 3절에서 자동 산출")

train (256351, 69) | test (90067, 68)
keep 로드: 없음 → 3절에서 자동 산출


## 3. NN 입력 파이프라인 (fold-내부 fit · 누수 안전)

In [3]:
TARGET="임신 성공 여부"; ID_COL="ID"
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
ORDINAL_COUNT_COLS=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수",
 "총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
COUNT_MAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
y=train[TARGET].astype(int).values

def prep_global(df):   # 타깃-무관 매핑(누수 아님)
    df=df.copy()
    for c in ORDINAL_COUNT_COLS:
        if c in df: df[c]=df[c].map(COUNT_MAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].map(m)
    return df

def nn_cols(tr, keep):
    if keep is None: keep=[c for c in tr.columns if c not in (TARGET,ID_COL)]
    keep=[c for c in keep if c in tr.columns and c not in (TARGET,ID_COL)]
    cat=[c for c in keep if not pd.api.types.is_numeric_dtype(tr[c])]
    num=[c for c in keep if c not in cat]
    return cat,num

def fit_fold(trdf, cat, num):
    enc={c:{v:i+1 for i,v in enumerate(trdf[c].astype(str).fillna("NA").value_counts().index)} for c in cat}
    med={c:float(trdf[c].median()) for c in num}
    mu={}; sd={}
    for c in num:
        x=trdf[c].fillna(med[c]).astype(float); mu[c]=float(x.mean()); sd[c]=float(x.std())+1e-6
    return enc,med,mu,sd

def transform(df, cat, num, enc, med, mu, sd):
    Xc=np.zeros((len(df),len(cat)),dtype=np.int64)
    for j,c in enumerate(cat):
        Xc[:,j]=df[c].astype(str).fillna("NA").map(enc[c]).fillna(0).astype(int).values
    Xn=np.zeros((len(df),len(num)*2),dtype=np.float32)
    for j,c in enumerate(num):
        Xn[:,2*j+1]=df[c].isna().astype(np.float32).values           # 결측 플래그
        x=df[c].fillna(med[c]).astype(float).values
        Xn[:,2*j]=((x-mu[c])/sd[c]).astype(np.float32)               # 스케일값
    return Xc,Xn

trg=prep_global(train); teg=prep_global(test)
# prune_list.json 없으면 순열 중요도로 keep 셋 즉석 산출 (T2 출력 의존 제거 · CPU ~수분)
if KEEP is None:
    print("prune_list.json 없음 → 순열 중요도로 keep 산출 (LGBM 홀드아웃, ~수분)...")
    import lightgbm as lgb
    Xp_=train.drop(columns=[TARGET,ID_COL]).copy()
    for c in Xp_.columns:
        if not pd.api.types.is_numeric_dtype(Xp_[c]): Xp_[c]=pd.factorize(Xp_[c].astype(str))[0]
    Xp_=Xp_.apply(pd.to_numeric,errors="coerce"); fp=list(Xp_.columns)
    a,b,ya,yb=train_test_split(Xp_,y,test_size=0.25,stratify=y,random_state=SEED)
    mp=lgb.train(dict(objective="binary",metric="auc",learning_rate=0.05,num_leaves=63,verbose=-1,seed=SEED),
                 lgb.Dataset(a,ya),num_boost_round=500,valid_sets=[lgb.Dataset(b,yb)],
                 callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(0)])
    bs=roc_auc_score(yb,mp.predict(b)); rp=np.random.default_rng(0); dr={}
    for c in fp:
        Xq=b.copy(); Xq[c]=Xq[c].values[rp.permutation(len(Xq))]; dr[c]=bs-roc_auc_score(yb,mp.predict(Xq))
    KEEP=[c for c in fp if dr[c]>=0.0001]
    import json as _json; _json.dump({"keep":KEEP,"prune":[c for c in fp if c not in KEEP]},open("prune_list.json","w"),ensure_ascii=False)
    print(f"  → keep={len(KEEP)}개 산출 (prune_list.json 저장)")
CAT,NUM=nn_cols(trg,KEEP)
print(f"입력: cat={len(CAT)}개(임베딩), num={len(NUM)}개(스케일+결측플래그={len(NUM)*2}차원)")

prune_list.json 없음 → 순열 중요도로 keep 산출 (LGBM 홀드아웃, ~수분)...
  → keep=34개 산출 (prune_list.json 저장)
입력: cat=6개(임베딩), num=28개(스케일+결측플래그=56차원)


## 4. 모델 — 임베딩-MLP + 학습/예측

In [4]:
def runr(p): return rankdata(p)/len(p)

class EmbMLP(nn.Module):
    def __init__(self, cat_dims, n_num, hidden=(256,128), p=0.2):
        super().__init__()
        self.embs=nn.ModuleList([nn.Embedding(d, min(50,(d+1)//2+1)) for d in cat_dims])
        emb_total=sum(e.embedding_dim for e in self.embs)
        layers=[]; din=emb_total+n_num
        for h in hidden: layers+=[nn.Linear(din,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(p)]; din=h
        layers+=[nn.Linear(din,1)]; self.mlp=nn.Sequential(*layers)
    def forward(self, xc, xn):
        if len(self.embs)>0:
            e=torch.cat([emb(xc[:,i]) for i,emb in enumerate(self.embs)],dim=1); x=torch.cat([e,xn],dim=1)
        else: x=xn
        return self.mlp(x).squeeze(1)

def train_nn(Xc_t,Xn_t,yt, Xc_v,Xn_v,yv, cat_dims, epochs, patience, bs, lr, hidden, dropout, seed=42, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE=="cuda": torch.cuda.manual_seed_all(seed)
    model=EmbMLP(cat_dims, Xn_t.shape[1], hidden, dropout).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    lossf=nn.BCEWithLogitsLoss()
    ds=TensorDataset(torch.as_tensor(Xc_t),torch.as_tensor(Xn_t),torch.as_tensor(yt,dtype=torch.float32))
    dl=DataLoader(ds,batch_size=bs,shuffle=True,drop_last=True)   # drop_last: BatchNorm 배치1 방지
    Xcv=torch.as_tensor(Xc_v).to(DEVICE); Xnv=torch.as_tensor(Xn_v).to(DEVICE)
    best=-1; best_state=None; bad=0
    for ep in range(epochs):
        model.train()
        for xc,xn,yb in dl:
            xc,xn,yb=xc.to(DEVICE),xn.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); loss=lossf(model(xc,xn),yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): pv=torch.sigmoid(model(Xcv,Xnv)).cpu().numpy()
        auc=roc_auc_score(yv,pv)
        if verbose: print(f"    ep{ep} val AUC={auc:.5f}")
        if auc>best+1e-5: best=auc; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    model.load_state_dict(best_state)
    return model, best

def predict_nn(model, Xc, Xn, bs=16384):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(Xc),bs):
            xc=torch.as_tensor(Xc[i:i+bs]).to(DEVICE); xn=torch.as_tensor(Xn[i:i+bs]).to(DEVICE)
            out.append(torch.sigmoid(model(xc,xn)).cpu().numpy())
    return np.concatenate(out)

## 5. Phase 0 — 프로토타입 (★ 풀런 전 게이트 P)

서브샘플·소수 epoch·단일 holdout으로 **싸게** 파이프라인+경쟁력 확인. **val AUC ≥ 0.72 통과 시에만** Phase 1.

> *Phase 0 < 0.72 시 1순위 의심: `특정 시술 유형`·`배아 생성 주요 이유`가 단일 고카디널리티 임베딩으로 들어가 트리가 쓰던 멀티핫 구조를 잃음 → 이 둘만 토큰 멀티핫으로 num 블록에 주면 보통 회복.*

In [5]:
t0=time.time()
idx=np.arange(len(trg))
sub_idx,_=train_test_split(idx, train_size=min(PROTO_N,len(trg)), stratify=y, random_state=SEED)
sd_=trg.iloc[sub_idx].reset_index(drop=True); ys=y[sub_idx]
tri,vai=train_test_split(np.arange(len(sd_)), test_size=0.2, stratify=ys, random_state=SEED)
enc,med,mu,sd=fit_fold(sd_.iloc[tri],CAT,NUM)
Xc_t,Xn_t=transform(sd_.iloc[tri],CAT,NUM,enc,med,mu,sd)
Xc_v,Xn_v=transform(sd_.iloc[vai],CAT,NUM,enc,med,mu,sd)
dims=[len(enc[c])+1 for c in CAT]
_,proto_auc=train_nn(Xc_t,Xn_t,ys[tri],Xc_v,Xn_v,ys[vai],dims,PROTO_EPOCHS,3,BATCH,LR,HIDDEN,DROPOUT,verbose=True)
PROTO_PASS = proto_auc>=PROTO_GATE
print(f"\n[Phase 0] 서브샘플 val AUC = {proto_auc:.5f} | 게이트 {PROTO_GATE} → {'통과 ✅ 풀런 진행' if PROTO_PASS else '보류 ⚠️ (버그/NN부적합 점검)'}  ({time.time()-t0:.0f}s)")

    ep0 val AUC=0.71369
    ep1 val AUC=0.72266
    ep2 val AUC=0.72286
    ep3 val AUC=0.72715
    ep4 val AUC=0.72689
    ep5 val AUC=0.72936
    ep6 val AUC=0.72973
    ep7 val AUC=0.73071

[Phase 0] 서브샘플 val AUC = 0.73071 | 게이트 0.72 → 통과 ✅ 풀런 진행  (9s)


## 6. Phase 1 — 풀 5-fold OOF (Phase 0 통과 시에만)

In [6]:
oof_nn=test_nn=None
if not PROTO_PASS:
    print("Phase 0 미통과 — Phase 1 건너뜀. 원인(버그/학습정체) 먼저 점검 후 재실행.")
else:
    folds=list(StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED).split(trg,y))
    oof_nn=np.zeros(len(trg)); test_nn=np.zeros(len(teg))
    for k,(tri,vai) in enumerate(folds):
        t1=time.time()
        enc,med,mu,sd=fit_fold(trg.iloc[tri],CAT,NUM)
        Xc_t,Xn_t=transform(trg.iloc[tri],CAT,NUM,enc,med,mu,sd)
        Xc_v,Xn_v=transform(trg.iloc[vai],CAT,NUM,enc,med,mu,sd)
        Xc_e,Xn_e=transform(teg,CAT,NUM,enc,med,mu,sd)
        dims=[len(enc[c])+1 for c in CAT]
        model,va=train_nn(Xc_t,Xn_t,y[tri],Xc_v,Xn_v,y[vai],dims,EPOCHS,PATIENCE,BATCH,LR,HIDDEN,DROPOUT,seed=SEED)
        oof_nn[vai]=predict_nn(model,Xc_v,Xn_v); test_nn+=predict_nn(model,Xc_e,Xn_e)/N_SPLITS
        print(f"  fold{k}: val AUC={va:.5f}  ({time.time()-t1:.0f}s)")
    print(f"\n[Phase 1] NN OOF AUC = {roc_auc_score(y,oof_nn):.5f}")

  fold0: val AUC=0.73511  (55s)
  fold1: val AUC=0.74062  (45s)
  fold2: val AUC=0.73849  (33s)
  fold3: val AUC=0.73509  (56s)
  fold4: val AUC=0.73910  (51s)

[Phase 1] NN OOF AUC = 0.73751


## 7. Phase 2 — 게이트 (단일 AUC · 트리 ρ · 증분 CI)

In [7]:
def get_tree_oof():
    for f,cols in [("oof_predictions.csv",["oof_lgb","oof_cat","oof_xgb"]),("oof_t2.csv",["oof_lgb_bin","oof_cat_bin","oof_xgb_bin"])]:
        for base in ["/kaggle/input/**/","","/kaggle/working/"]:
            g=glob.glob(base+f,recursive=True)
            if g:
                od=pd.read_csv(g[0])
                if len(od)==len(train) and all(c in od.columns for c in cols):
                    print("[트리 OOF 로드]",g[0]); return {m:od[c].values for m,c in zip(["lgb","cat","xgb"],cols)}
    return None

if PROTO_PASS and oof_nn is not None:
    auc_nn=roc_auc_score(y,oof_nn)
    trees=get_tree_oof()
    if trees is None:
        print("⚠️ 트리 OOF 파일 없음 — 이전 노트북의 oof_predictions.csv/oof_t2.csv를 Add Input 하세요(블렌드 게이트 생략).")
    else:
        rho=max(abs(pd.Series(runr(oof_nn)).corr(pd.Series(runr(trees[m])),method="spearman")) for m in trees)
        print(f"단일 NN OOF AUC = {auc_nn:.5f}  (게이트 ≥{GATE_AUC} → {'OK' if auc_nn>=GATE_AUC else 'FAIL'})")
        print(f"트리와 최대 Spearman = {rho:.3f}  (게이트 ≤{GATE_RHO} → {'OK(decorrelated)' if rho<=GATE_RHO else 'FAIL(중복)'})")
        # 블렌드 증분
        allm={**{m:runr(trees[m]) for m in trees}, "nn":runr(oof_nn)}
        def hill(d,yy,n=80):
            nm=list(d); s0={n_:roc_auc_score(yy,d[n_]) for n_ in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
            for _ in range(n):
                cb,ca=None,-1
                for n_ in nm:
                    a=roc_auc_score(yy,(s+d[n_])/(len(ens)+1))
                    if a>ca: ca,cb=a,n_
                ens.append(cb); s=s+d[cb]
                if ca>best[1]: best=(list(ens),ca)
            from collections import Counter; c=Counter(best[0]); return {n_:c.get(n_,0)/len(best[0]) for n_ in nm}, best[1]
        base3={m:runr(trees[m]) for m in trees}; w3,bl3=hill(base3,y); blend3=sum(w3[m]*base3[m] for m in w3)
        wN,blN=hill(allm,y); blendN=sum(wN[m]*allm[m] for m in wN)
        rng=np.random.default_rng(0); ix=np.arange(len(y)); ds=[]
        for _ in range(BOOT_N):
            s=rng.choice(ix,len(ix),replace=True); ds.append(roc_auc_score(y[s],blendN[s])-roc_auc_score(y[s],blend3[s]))
        lo,hi=np.percentile(ds,[2.5,97.5])
        print(f"\n3트리 블렌드={bl3:.5f} | +NN={blN:.5f} | Δ={blN-bl3:+.5f} | nn가중치={wN['nn']:.3f}")
        print(f"증분 95%CI=[{lo:+.5f},{hi:+.5f}] → {'채택 후보(멀티시드+LB로 확정)' if lo>0 else '0 포함=천장 확정 네거티브'}")
        adopt = (auc_nn>=GATE_AUC) and (rho<=GATE_RHO) and (lo>0)
        print(f"\n종합 게이트: {'★ 채택 후보 — 멀티시드(8절)→LB 1회' if adopt else '네거티브 — 계열 다른 NN으로도 불변=정보천장 확정(발표 자산)'}")

[트리 OOF 로드] /kaggle/input/notebooks/jaeranlee/subfertility-predict-ai-3tree-lr/oof_predictions.csv
단일 NN OOF AUC = 0.73751  (게이트 ≥0.735 → OK)
트리와 최대 Spearman = 0.973  (게이트 ≤0.93 → FAIL(중복))

3트리 블렌드=0.74012 | +NN=0.74044 | Δ=+0.00032 | nn가중치=0.255
증분 95%CI=[+0.00017,+0.00047] → 채택 후보(멀티시드+LB로 확정)

종합 게이트: 네거티브 — 계열 다른 NN으로도 불변=정보천장 확정(발표 자산)


## 8. (선택) 멀티시드 — `RUN_MULTISEED=True` & 채택 후보일 때만

NN은 시드분산이 크므로 채택 직전 2~3시드 재학습해 증분 mean±std 확인.

In [8]:
if RUN_MULTISEED and PROTO_PASS and oof_nn is not None and 'trees' in dir() and trees is not None:
    folds=list(StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED).split(trg,y))
    seed_aucs=[]
    for sd_seed in (42,7,2024):
        on=np.zeros(len(trg))
        for tri,vai in folds:
            enc,med,mu,sd=fit_fold(trg.iloc[tri],CAT,NUM)
            Xc_t,Xn_t=transform(trg.iloc[tri],CAT,NUM,enc,med,mu,sd); Xc_v,Xn_v=transform(trg.iloc[vai],CAT,NUM,enc,med,mu,sd)
            dims=[len(enc[c])+1 for c in CAT]
            m,_=train_nn(Xc_t,Xn_t,y[tri],Xc_v,Xn_v,y[vai],dims,EPOCHS,PATIENCE,BATCH,LR,HIDDEN,DROPOUT,seed=sd_seed)
            on[vai]=predict_nn(m,Xc_v,Xn_v)
        seed_aucs.append(roc_auc_score(y,on)); print(f"  seed{sd_seed}: NN OOF={seed_aucs[-1]:.5f}")
    print(f"NN OOF mean±std = {np.mean(seed_aucs):.5f} ± {np.std(seed_aucs):.5f}  (Δ 판정은 이 std 기준)")
else:
    print("멀티시드 생략(RUN_MULTISEED=False 또는 채택후보 아님).")

  seed42: NN OOF=0.73751
  seed7: NN OOF=0.73734
  seed2024: NN OOF=0.73723
NN OOF mean±std = 0.73736 ± 0.00011  (Δ 판정은 이 std 기준)


## 9. 산출물

In [9]:
def get_tree_test():
    for f,cols in [("test_predictions.csv",["test_lgb","test_cat","test_xgb"]),
                   ("test_t2.csv",["test_lgb_bin","test_cat_bin","test_xgb_bin"])]:
        for base in ["/kaggle/input/**/","","/kaggle/working/"]:
            gp=glob.glob(base+f,recursive=True)
            if gp:
                td=pd.read_csv(gp[0])
                if len(td)==len(test) and all(c in td.columns for c in cols):
                    print("[트리 test 예측 로드]",gp[0]); return {m:td[c].values for m,c in zip(["lgb","cat","xgb"],cols)}
    return None

if PROTO_PASS and oof_nn is not None:
    pd.DataFrame({"oof_nn":oof_nn,"y":y}).to_csv("oof_nn.csv",index=False)
    pd.DataFrame({"test_nn":test_nn,"ID":test[ID_COL].values}).to_csv("test_nn.csv",index=False)
    print("oof_nn.csv / test_nn.csv 저장 (트리와 fold 정렬, 블렌드/스태킹 재사용용)")
    # ★ 채택 후보일 때만 블렌드 제출 — test 트리 예측(OOF 아님)으로 길이(90,067) 일치
    if 'wN' in dir() and 'adopt' in dir() and adopt:
        tt=get_tree_test()
        if tt is not None:
            test_all={**{m:runr(tt[m]) for m in tt}, "nn":runr(test_nn)}   # 전부 test 길이
            s=sample_sub.copy() if sample_sub is not None else pd.DataFrame({ID_COL:test[ID_COL].values})
            pc=[c for c in s.columns if c!=ID_COL]; pc=pc[0] if pc else "probability"
            s[pc]=sum(wN[m]*test_all[m] for m in wN if m in test_all)
            s.to_csv("submission_nn_blend.csv",index=False)
            print("submission_nn_blend.csv 저장 (게이트 통과 → 멀티시드+LB 1회로 최종 확정)")
        else:
            print("→ 게이트 통과지만 test_predictions.csv/test_t2.csv 없음 → 제출 생략. Add Input 후 기존 블렌드 노트북서 합치면 됨.")
    else:
        print("→ 게이트 미통과/미확정: 제출 생성 안 함. OOF/test만 저장(채택 확정 후 수동 제출).")
else:
    print("Phase 1 미실행 — 산출물 없음.")

oof_nn.csv / test_nn.csv 저장 (트리와 fold 정렬, 블렌드/스태킹 재사용용)
→ 게이트 미통과/미확정: 제출 생성 안 함. OOF/test만 저장(채택 확정 후 수동 제출).


In [10]:
import pandas as pd, numpy as np, glob
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
def rr(p): return rankdata(p)/len(p)

od=pd.read_csv(glob.glob("/kaggle/input/**/oof_predictions.csv",recursive=True)[0]); y=od["y"].values
tree_oof=rr((rr(od["oof_lgb"])+rr(od["oof_cat"])+rr(od["oof_xgb"]))/3)
nn_oof=rr(pd.read_csv("oof_nn.csv")["oof_nn"].values)
best=(1.0,0)
for w in np.linspace(0,1,101):
    a=roc_auc_score(y, w*tree_oof+(1-w)*nn_oof)
    if a>best[1]: best=(w,a)
w=best[0]
print(f"OOF 3트리={roc_auc_score(y,tree_oof):.5f} → +NN best(트리 w={w:.2f})={best[1]:.5f} | Δ={best[1]-roc_auc_score(y,tree_oof):+.5f}")

b4=pd.read_csv(glob.glob("/kaggle/input/**/submission_blend4.csv",recursive=True)[0])
idc=[c for c in b4.columns if c.lower()=="id"][0]; pc=[c for c in b4.columns if c.lower()!="id"][0]
tn=pd.read_csv("test_nn.csv")
sub=pd.DataFrame({idc:b4[idc].values})
sub[pc]=w*rr(b4[pc].values)+(1-w)*rr(tn["test_nn"].values)
sub.to_csv("submission_nn_blend.csv",index=False)
print("submission_nn_blend.csv:",sub.shape,"| 결측",int(sub[pc].isna().sum()))

OOF 3트리=0.74010 → +NN best(트리 w=0.75)=0.74043 | Δ=+0.00033
submission_nn_blend.csv: (90067, 2) | 결측 0


## 10. 결정

- **채택:** 단일 OOF≥0.735 **AND** ρ≤0.93 **AND** 증분 CI>0 **AND** 멀티시드 Δ>std **AND** LB 1회 재현 → NN 멤버 합류, 운영 블렌드 갱신.
- **네거티브(천장 확정):** ρ>0.93(트리와 중복) or 단독<0.735(약함) or 증분 CI 0 포함 → **"계열 다른 NN으로도 불변 = 정보천장 확정"** — 가장 강한 발표 논거.
- 운영 제출은 **3트리 블렌드(0.74182) 고정.** NN은 전 게이트 통과 시에만.

> **돌리는 순서:** Phase 0(프로토타입) 먼저 → 통과 시 Phase 1 자동. ★ GPU 세션 + 이전 OOF(oof_predictions.csv/oof_t2.csv)·prune_list.json을 **Add Input**. Phase 0에서 막히면 풀런 전 중단해 4시간 낭비 방지.